# Unlearning Prompt A/B Test Pipeline

This notebook runs a controlled A/B test of four prompt strategies for classifying whether each paragraph in the **GPT Test** sheet is **Unlearning = Yes/No**.

It is designed for the uploaded workbook **`Unlearning Codebook.xlsx`** and supports three API providers:

1. OpenAI / ChatGPT API
2. Anthropic Claude API
3. Google Gemini API

The pipeline:

- Loads `Examples`, `GPT Test`, `Codebook`, and `Decision Rules` sheets.
- Expands the `Codes` column into parsed metadata columns and optional one-hot code indicator columns.
- Builds four prompt variants:
  - `direct_no_context`
  - `definitions_only`
  - `definitions_examples_no_metadata`
  - `definitions_examples_with_metadata`
- Sends each prompt/test paragraph pair to the selected LLMs.
- Parses JSON responses.
- Compares predicted `Unlearning` labels with ground truth.
- Reports accuracy, precision, recall, F1, token usage, estimated/actual cost, and total spend by prompt and model.
- Saves long-form predictions and summary reports.

> Keep `RUN_API_CALLS = False` first to preview prompts and estimated cost. Set it to `True` only after confirming the estimated budget.

In [ ]:
# Optional: install dependencies once. Uncomment if needed.
%pip install -q pandas openpyxl tqdm tenacity python-dotenv tiktoken openai anthropic google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 20.5 MB/s eta 0:00:00


## 1. Configuration

Set your workbook path, model choices, budget limit, and API execution switch here.

API keys should be stored as environment variables or in a local `.env` file:

```bash
OPENAI_API_KEY=...
ANTHROPIC_API_KEY=...
GEMINI_API_KEY=...
```

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import re
import json
import time
import math
import hashlib
from collections import defaultdict


import pandas as pd
import numpy as np
from IPython.display import display
try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None, **kwargs):
        return iterable if iterable is not None else range(total or 0)

load_dotenv(dotenv_path=Path(".env"))

# Workbook location. Keep the Excel file in the same folder as this notebook,
# or change this path to wherever your workbook is stored.
WORKBOOK_PATH = Path("/content/Unlearning Codebook.xlsx")
if not WORKBOOK_PATH.exists():
    WORKBOOK_PATH = Path("/content/Unlearning Codebook.xlsx")

OUTPUT_DIR = Path("outputs_unlearning_ab_test")
OUTPUT_DIR.mkdir(exist_ok=True)

# First run should be dry-run only. Set True when ready to spend API credits.
RUN_API_CALLS = True

# Use a small row limit while testing. Set to None to run the full GPT Test sheet.
ROW_LIMIT = None

# Use all labeled examples by default. For budget tests, try 8, 12, or 20.
MAX_FEWSHOT_EXAMPLES = None

# Optional truncation to control cost for very long paragraphs. Set None for no truncation.
MAX_TEXT_CHARS = None

# API call settings
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 250
REQUEST_SLEEP_SECONDS = 0.2
RESUME_FROM_JSONL = True

# Hard budget guard. The notebook will stop before API calls if estimated grid cost is above this.
MAX_ESTIMATED_COST_USD = 5.00

# Pricing is USD per 1M tokens. Update this dictionary if your selected model/pricing changes.
# These defaults are intentionally low-cost models suitable for annotation A/B tests.
MODEL_CONFIG = [
    {
        "provider": "openai",
        "model": "gpt-5.4-nano",
        "enabled": True,
        "api_key_env": "OPENAI_API_KEY",
        "input_usd_per_1m": 0.20,
        "output_usd_per_1m": 1.25,
        "request_sleep_seconds": 0.2,
    },
    {
        "provider": "anthropic",
        "model": "claude-haiku-4-5-20251001",
        "enabled": True,
        "api_key_env": "ANTHROPIC_API_KEY",
        "input_usd_per_1m": 1.00,
        "output_usd_per_1m": 5.00,
        "request_sleep_seconds": 0.2,
    },
    {
        "provider": "gemini",
        "model": "gemini-2.5-flash-lite",
        "enabled": True,
        "api_key_env": "GEMINI_API_KEY",
        "input_usd_per_1m": 0.10,
        "output_usd_per_1m": 0.40,
        "request_sleep_seconds": 7.0,

    },
]

PROMPT_VARIANTS_TO_RUN = [
    "direct_no_context",
    "definitions_only",
    "definitions_examples_no_metadata",
    "definitions_examples_with_metadata",
]

print("Workbook path:", WORKBOOK_PATH)
print("Output folder:", OUTPUT_DIR.resolve())

Workbook path: /content/Unlearning Codebook.xlsx
Output folder: /content/outputs_unlearning_ab_test


## 2. Load workbook sheets

The workbook is expected to contain these sheets:

- `Examples`
- `GPT Test`
- `Codebook`
- `Decision Rules`

In [ ]:
def drop_empty_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed", na=False)]
    return df

examples_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="Examples"))
test_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="GPT Test"))
raw_codebook_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Codebook", header=None)
decision_rules_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="Decision Rules"))

# Extract Codebook objective and table. The first row is the objective; the real header row starts with "Code".
objective_text = str(raw_codebook_df.iloc[0, 0]) if not pd.isna(raw_codebook_df.iloc[0, 0]) else ""
header_candidates = raw_codebook_df.index[
    raw_codebook_df.iloc[:, 0].astype(str).str.strip().str.lower().eq("code")
].tolist()
if not header_candidates:
    raise ValueError("Could not find the Codebook table header row where column A equals 'Code'.")

codebook_header_row = header_candidates[0]
codebook_df = raw_codebook_df.iloc[codebook_header_row + 1:].copy()
codebook_df.columns = [str(x).strip() if not pd.isna(x) else "" for x in raw_codebook_df.iloc[codebook_header_row].tolist()]
codebook_df = codebook_df.dropna(axis=0, how="all")
codebook_df = codebook_df.loc[:, [c for c in codebook_df.columns if c != ""]]
codebook_df = codebook_df.reset_index(drop=True)

print("Examples shape:", examples_df.shape)
print("GPT Test shape:", test_df.shape)
print("Codebook table shape:", codebook_df.shape)
print("Decision Rules shape:", decision_rules_df.shape)

display(examples_df.head(3))
display(test_df.head(3))
display(codebook_df.head(8))

Examples shape: (62, 7)
GPT Test shape: (11, 8)
Codebook table shape: (13, 6)
Decision Rules shape: (4, 7)


,Number,Reference,Text Content,Rationale,Source,Unlearning,Codes
0,1:1,p 3,"Coordination within EPA, with State and local ...",Not unlearning. As no logic being abandoned. o...,EPA,No,"Depth: Moderate depth, Nature: Technical Natur..."
1,1:2,pp 9 – 10,"EPA officials told us that, in some instances,...",Not unlearning. As no logic being abandoned ju...,EPA,No,"Nature: Technical Nature, Pillar: Merging (Int..."
2,1:3,p 11,"From our review of wastewater issues, we found...",Unlearning as it states EPA was not commuunica...,EPA,Yes,"Depth: Low depth, Nature: Technical Nature, Pi..."


,Number,Text Content,Document,Codes,Unlearning,Pillar,Depth,Nature
0,14:2,Because of FEMA’s mission performance during H...,gao-06-442t.pdf,"Depth: High depth, Nature: Technical Nature, P...",Yes,Discarding,High,Technical
1,14:3,"In addition, we have done a great deal of work...",gao-06-442t.pdf,"Depth: High depth, Nature: Technical Nature, P...",Yes,Discarding,High,Technical
2,14:4,Because of FEMA’s mission performance during H...,gao-06-442t.pdf,"Depth: High depth, Pillar: Discarding (Normati...",Yes,Discarding,High,NaN


,Code,Definition,Detection Logic,Examples,Positive Clarification,Negative Clarification
0,Secondary Data from US govt agencies evaluatio...,"Text drawn from formal evaluations, audits, or...","The ""Subtractive"" Test",Passages that speak to unlearning framework,Step 1: Identify if Unlearning exists.\nStep 2...,NaN
1,Binary classification based on unlearning defi...,NaN,NaN,NaN,NaN,NaN
2,Unlearning,Unlearning refers to the deliberate process by...,Does this text advocate for change that implie...,"As the White House report states, “Ultimately,...",Code when the text explicitly calls for:\nAban...,"""Do NOT code if:\nThe text only discusses “les..."
3,Unlearning typology (Primary Analytic Codes),NaN,NaN,NaN,NaN,NaN
4,Reconsidering (Epistemic Unlearning),This involves the unlearning of dominant knowl...,Are they questioning the truth or authority of...,"""The question of how and when an event becomes...",Code when the text:\nQuestions how presidentia...,Do NOT code if:\nThe text proposes a technical...
5,Discarding (Normative Unlearning),This entails the rejection and abandonment of ...,Are they terminating a policy or law or organi...,"""removing statutory restrictions on DOD’s auth...",Code when the text:\n\nCalls for eliminating s...,"Do NOT code if:\nThe text suggests reform, adj..."
6,Realignment (Technical Unlearning),This pillar addresses the obsolescence or over...,Are they admitting an engineering or technical...,"""Andrew and Hugo, we identified the need for t...",Code when the text:\nCalls for revising engine...,Do not code if it is a routine maintenance upd...
7,Merging (Integrative Unlearning),This pillar rests on overcoming disciplinary a...,Are they destroying silos to integrate knowledge?,"""Such operational plans should, for example, f...",Code when the text:\nExplicitly calls for join...,Do NOT code if:\nCollaboration is mentioned sy...


## 3. Preprocess labels and expand the `Codes` column

This section parses values such as:

`Depth: Moderate depth, Nature: Technical Nature, Pillar: Merging (Integrative), Stakeholder: EPA, Unlearning`

into structured columns like:

- `code_depth`
- `code_nature`
- `code_pillar`
- `code_stakeholder`
- `code_unlearning`

It also creates optional one-hot indicator columns such as `code_pillar__merging_integrative`.

In [ ]:
def normalize_yes_no(value):
    """Normalize yes/no-like values to 'Yes', 'No', or np.nan."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if s in {"yes", "y", "true", "1", "unlearning"}:
        return "Yes"
    if s in {"no", "n", "false", "0", "not unlearning", "non-unlearning"}:
        return "No"
    if "yes" == s[:3]:
        return "Yes"
    if "no" == s[:2]:
        return "No"
    return np.nan


def clean_text_for_prompt(text, max_chars=MAX_TEXT_CHARS):
    if pd.isna(text):
        return ""
    s = re.sub(r"\s+", " ", str(text)).strip()
    if max_chars is not None and len(s) > max_chars:
        s = s[:max_chars].rstrip() + " ... [TRUNCATED]"
    return s


def slugify(value):
    s = str(value).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_") or "blank"


def parse_codes_cell(cell):
    """Parse comma-separated code metadata into a dict of lists."""
    parsed = defaultdict(list)
    if pd.isna(cell):
        return parsed
    text = str(cell).strip()
    if not text:
        return parsed

    parts = [p.strip() for p in text.split(",") if p.strip()]
    for part in parts:
        if ":" in part:
            key, value = part.split(":", 1)
            key = slugify(key)
            value = value.strip()
            if value and value not in parsed[key]:
                parsed[key].append(value)
        else:
            # Handles standalone values such as "Unlearning"
            key = slugify(part)
            if key == "unlearning":
                parsed["unlearning"].append("Yes")
            else:
                parsed["flag"].append(part)
    return parsed


def expand_codes_column(df: pd.DataFrame, code_col: str = "Codes", add_indicators: bool = True) -> pd.DataFrame:
    df = df.copy()
    if code_col not in df.columns:
        return df

    parsed_rows = [parse_codes_cell(v) for v in df[code_col]]
    all_keys = sorted({k for row in parsed_rows for k in row.keys()})

    for key in all_keys:
        df[f"code_{key}"] = ["; ".join(row.get(key, [])) for row in parsed_rows]

    if add_indicators:
        indicator_values = sorted({
            (key, value)
            for row in parsed_rows
            for key, values in row.items()
            for value in values
        })
        for key, value in indicator_values:
            col = f"code_{key}__{slugify(value)}"
            df[col] = [int(value in row.get(key, [])) for row in parsed_rows]

    return df


examples_df = expand_codes_column(examples_df, "Codes", add_indicators=True)
test_df = expand_codes_column(test_df, "Codes", add_indicators=True)

if "Unlearning" not in test_df.columns:
    raise ValueError("GPT Test sheet must contain a ground-truth 'Unlearning' column.")
if "Text Content" not in test_df.columns:
    raise ValueError("GPT Test sheet must contain a 'Text Content' column.")
if "Text Content" not in examples_df.columns or "Unlearning" not in examples_df.columns:
    raise ValueError("Examples sheet must contain 'Text Content' and 'Unlearning' columns.")

test_df["ground_truth_unlearning"] = test_df["Unlearning"].apply(normalize_yes_no)
examples_df["example_unlearning"] = examples_df["Unlearning"].apply(normalize_yes_no)

test_df = test_df.dropna(subset=["Text Content", "ground_truth_unlearning"]).reset_index(drop=True)
examples_df = examples_df.dropna(subset=["Text Content", "example_unlearning"]).reset_index(drop=True)

if ROW_LIMIT is not None:
    test_df = test_df.head(ROW_LIMIT).copy()

print("Clean examples:", examples_df.shape)
print("Clean test rows:", test_df.shape)
print("Expanded code columns in GPT Test:")
print([c for c in test_df.columns if c.startswith("code_")][:30])

display(test_df[["Number", "ground_truth_unlearning", "code_depth", "code_nature", "code_pillar"]].head())

Clean examples: (6, 30)
Clean test rows: (11, 26)
Expanded code columns in GPT Test:
['code_depth', 'code_nature', 'code_pillar', 'code_stakeholder', 'code_unlearning', 'code_depth__high_depth', 'code_depth__moderate_depth', 'code_nature__normative_nature', 'code_nature__technical_nature', 'code_pillar__discarding_normative', 'code_pillar__merging_integrative', 'code_pillar__reconsidering_epistemic', 'code_stakeholder__congress', 'code_stakeholder__dept_of_defense', 'code_stakeholder__fema', 'code_stakeholder__miscelleanous', 'code_unlearning__yes']


,Number,ground_truth_unlearning,code_depth,code_nature,code_pillar
0,14:2,Yes,High depth,Technical Nature,Discarding (Normative)
1,14:3,Yes,High depth,Technical Nature,Discarding (Normative)
2,14:4,Yes,High depth,,Discarding (Normative)
3,14:5,Yes,Moderate depth,Normative Nature,Reconsidering (Epistemic)
4,14:6,Yes,Moderate depth,Normative Nature,Reconsidering (Epistemic)


## 4. Build reusable prompt context

The `definitions_only` prompt uses the Codebook but deliberately excludes the Codebook's `Examples` column so it remains a definitions-only condition.

The two few-shot prompts use the `Examples` sheet:

- no metadata: `Text Content` + `Unlearning`
- with metadata: `Text Content` + `Unlearning` + `Rationale` + `Source` + `Codes` + expanded code fields

In [ ]:
def choose_examples(df: pd.DataFrame, max_examples=MAX_FEWSHOT_EXAMPLES, seed=42) -> pd.DataFrame:
    """Use all examples by default; if max_examples is set, sample in a label-balanced way."""
    if max_examples is None or max_examples >= len(df):
        return df.copy()

    rng = np.random.default_rng(seed)
    groups = []
    labels = ["Yes", "No"]
    per_label = max(1, max_examples // len(labels))

    for label in labels:
        sub = df[df["example_unlearning"] == label]
        if len(sub) == 0:
            continue
        take = min(per_label, len(sub))
        groups.append(sub.sample(n=take, random_state=seed))

    selected = pd.concat(groups, ignore_index=False) if groups else df.head(0)

    # Fill remaining slots if needed.
    remaining = max_examples - len(selected)
    if remaining > 0:
        rest = df.drop(index=selected.index, errors="ignore")
        if len(rest) > 0:
            selected = pd.concat([selected, rest.sample(n=min(remaining, len(rest)), random_state=seed)])

    return selected.sort_index().reset_index(drop=True)


prompt_examples_df = choose_examples(examples_df, MAX_FEWSHOT_EXAMPLES)
print(f"Using {len(prompt_examples_df)} examples in few-shot prompts.")


def make_codebook_definitions_text(codebook: pd.DataFrame, objective: str = "") -> str:
    keep_cols = [
        "Code",
        "Definition",
        "Detection Logic",
        "Positive Clarification",
        "Negative Clarification",
    ]
    available_cols = [c for c in keep_cols if c in codebook.columns]
    rows = []

    if objective:
        rows.append("Research objective:\n" + clean_text_for_prompt(objective, max_chars=None))

    rows.append("Codebook definitions and detection guidance:")
    for _, row in codebook.iterrows():
        code_name = clean_text_for_prompt(row.get("Code", ""), max_chars=None)
        if not code_name or code_name.lower() == "nan":
            continue
        filled = []
        for col in available_cols:
            val = clean_text_for_prompt(row.get(col, ""), max_chars=None)
            if val and val.lower() != "nan":
                filled.append(f"{col}: {val}")
        if filled:
            rows.append("\n- " + "\n  ".join(filled))
    return "\n".join(rows)


def make_examples_text(examples: pd.DataFrame, include_metadata: bool) -> str:
    blocks = []
    for i, (_, row) in enumerate(examples.iterrows(), start=1):
        text = clean_text_for_prompt(row.get("Text Content", ""), max_chars=MAX_TEXT_CHARS)
        label = normalize_yes_no(row.get("Unlearning", row.get("example_unlearning", "")))
        block = [f"Example {i}:", f"Text Content: {text}", f"Unlearning: {label}"]

        if include_metadata:
            metadata_cols = [
                "Number",
                "Reference",
                "Rationale",
                "Source",
                "Codes",
                "code_depth",
                "code_nature",
                "code_pillar",
                "code_stakeholder",
                "code_unlearning",
            ]
            for col in metadata_cols:
                if col in row.index:
                    val = row.get(col, "")
                    if not pd.isna(val) and str(val).strip():
                        block.append(f"{col}: {clean_text_for_prompt(val, max_chars=None)}")
        blocks.append("\n".join(block))
    return "\n\n".join(blocks)


CODEBOOK_DEFINITIONS_TEXT = make_codebook_definitions_text(codebook_df, objective_text)
EXAMPLES_TEXT_NO_METADATA = make_examples_text(prompt_examples_df, include_metadata=False)
EXAMPLES_TEXT_WITH_METADATA = make_examples_text(prompt_examples_df, include_metadata=True)

print("Definitions context characters:", len(CODEBOOK_DEFINITIONS_TEXT))
print("Examples without metadata characters:", len(EXAMPLES_TEXT_NO_METADATA))
print("Examples with metadata characters:", len(EXAMPLES_TEXT_WITH_METADATA))

Using 6 examples in few-shot prompts.
Definitions context characters: 9205
Examples without metadata characters: 5642
Examples with metadata characters: 7811


## 5. Prompt variants

Each variant asks for the same JSON output schema:

```json
{
  "unlearning": "Yes or No",
  "confidence": 0.0,
  "rationale": "brief reason"
}
```

Only the context changes between variants.

In [ ]:
BASE_SYSTEM_PROMPT = """
You are a careful research annotation assistant.
Return only valid JSON. Do not include markdown, commentary, or extra keys.
""".strip()

JSON_OUTPUT_INSTRUCTION = 'Return JSON only: {"unlearning":"Yes or No","confidence":0.0,"rationale":"brief one-sentence reason"}'

# Each variant gets only the information it is supposed to receive.
# This keeps the A/B test cleaner and avoids spending tokens on negative instructions
# such as "do not use examples" when no examples are actually being supplied.
PROMPT_CONTEXT_DIRECT_NO_CONTEXT = ""
PROMPT_CONTEXT_DEFINITIONS_ONLY = CODEBOOK_DEFINITIONS_TEXT
PROMPT_CONTEXT_DEFINITIONS_EXAMPLES_NO_METADATA = f"""
{CODEBOOK_DEFINITIONS_TEXT}

Labeled examples:
{EXAMPLES_TEXT_NO_METADATA}
""".strip()
PROMPT_CONTEXT_DEFINITIONS_EXAMPLES_WITH_METADATA = f"""
{CODEBOOK_DEFINITIONS_TEXT}

Labeled examples:
{EXAMPLES_TEXT_WITH_METADATA}
""".strip()

PROMPT_CONTEXT_BY_VARIANT = {
    "direct_no_context": PROMPT_CONTEXT_DIRECT_NO_CONTEXT,
    "definitions_only": PROMPT_CONTEXT_DEFINITIONS_ONLY,
    "definitions_examples_no_metadata": PROMPT_CONTEXT_DEFINITIONS_EXAMPLES_NO_METADATA,
    "definitions_examples_with_metadata": PROMPT_CONTEXT_DEFINITIONS_EXAMPLES_WITH_METADATA,
}

TASK_BY_VARIANT = {
    "direct_no_context": "Is this passage an example of unlearning? Answer Yes or No.",
    "definitions_only": "Using the definitions, classify whether this passage is Unlearning.",
    "definitions_examples_no_metadata": "Using the definitions and labeled examples, classify whether this passage is Unlearning.",
    "definitions_examples_with_metadata": "Using the definitions and labeled examples, classify whether this passage is Unlearning.",
}


def build_prompt_messages(variant_name: str, test_row: pd.Series) -> list[dict]:
    """Build compact messages for one prompt variant and one GPT Test row."""
    if variant_name not in PROMPT_CONTEXT_BY_VARIANT:
        raise ValueError(f"Unknown prompt variant: {variant_name}")

    text = clean_text_for_prompt(test_row["Text Content"], max_chars=MAX_TEXT_CHARS)
    number = test_row.get("Number", test_row.name)
    context = PROMPT_CONTEXT_BY_VARIANT[variant_name]

    prompt_parts = [TASK_BY_VARIANT[variant_name]]
    if context:
        prompt_parts.append(context)
    prompt_parts.extend([
        f"Passage ID: {number}",
        f"Passage:\n{text}",
        JSON_OUTPUT_INSTRUCTION,
    ])

    user_prompt = "\n\n".join(part for part in prompt_parts if str(part).strip()).strip()

    return [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


# Preview one prompt from each variant.
preview_row = test_df.iloc[0]
for variant in PROMPT_VARIANTS_TO_RUN:
    messages = build_prompt_messages(variant, preview_row)
    print("=" * 100)
    print("PROMPT VARIANT:", variant)
    print(messages[1]["content"][:2500])
    print("...\n")



PROMPT VARIANT: direct_no_context
Is this passage an example of unlearning? Answer Yes or No.

Passage ID: 14:2

Passage:
Because of FEMA’s mission performance during Hurricane Katrina, concerns have been raised regarding the agency’s organizational placement, including whether it should be disbanded and functions moved to other agencies, remain within the Department of Homeland Security, or become an independent agency. However, other factors such as leadership and resources may be more important to FEMA’s future success than organizational placement.

Return JSON only: {"unlearning":"Yes or No","confidence":0.0,"rationale":"brief one-sentence reason"}
...

PROMPT VARIANT: definitions_only
Using the definitions, classify whether this passage is Unlearning.

Research objective:
Objective: The key objective of this paper is "how to study unlearning?" This codebook supports this objective through the development of an empirical, and evidence-based understanding of how unlearning shows up

## 6. Token and cost estimation

Actual token usage comes from each API response after you run the model. Before spending credits, this section estimates cost using prompt length and model pricing.

In [ ]:
def estimate_tokens_text(text: str, provider: str = "generic") -> int:
    """Best-effort token estimate for preflight budgeting."""
    if text is None:
        return 0
    text = str(text)
    if provider == "openai":
        try:
            import tiktoken
            enc = tiktoken.get_encoding("cl100k_base")
            return len(enc.encode(text))
        except Exception:
            pass
    # Simple fallback used for Claude/Gemini preflight estimates.
    return max(1, math.ceil(len(text) / 4))


def estimate_messages_tokens(messages: list[dict], provider: str = "generic") -> int:
    # Add small overhead per message.
    return sum(estimate_tokens_text(m.get("content", ""), provider) + 6 for m in messages) + 10


def cost_from_tokens(input_tokens: int, output_tokens: int, cfg: dict) -> float:
    return (
        (input_tokens / 1_000_000) * cfg["input_usd_per_1m"]
        + (output_tokens / 1_000_000) * cfg["output_usd_per_1m"]
    )


def active_model_configs(require_api_key: bool = False) -> list[dict]:
    active = []
    for cfg in MODEL_CONFIG:
        if not cfg.get("enabled", True):
            continue
        key_present = bool(os.getenv(cfg.get("api_key_env", "")))
        if require_api_key and not key_present:
            print(f"Skipping {cfg['provider']} / {cfg['model']} because {cfg['api_key_env']} is not set.")
            continue
        active.append(cfg)
    return active


def estimate_full_grid(test_rows: pd.DataFrame, model_configs: list[dict], variants: list[str], assumed_output_tokens=80) -> pd.DataFrame:
    rows = []
    for cfg in model_configs:
        for variant in variants:
            input_total = 0
            for _, row in test_rows.iterrows():
                messages = build_prompt_messages(variant, row)
                input_total += estimate_messages_tokens(messages, provider=cfg["provider"])
            output_total = assumed_output_tokens * len(test_rows)
            rows.append({
                "provider": cfg["provider"],
                "model": cfg["model"],
                "prompt_variant": variant,
                "n_rows": len(test_rows),
                "estimated_input_tokens": int(input_total),
                "assumed_output_tokens": int(output_total),
                "estimated_total_tokens": int(input_total + output_total),
                "estimated_cost_usd": cost_from_tokens(input_total, output_total, cfg),
            })
    return pd.DataFrame(rows)


estimate_df = estimate_full_grid(test_df, active_model_configs(require_api_key=False), PROMPT_VARIANTS_TO_RUN)
display(estimate_df)

provider_estimates = estimate_df.groupby(["provider", "model"], as_index=False).agg(
    estimated_input_tokens=("estimated_input_tokens", "sum"),
    assumed_output_tokens=("assumed_output_tokens", "sum"),
    estimated_total_tokens=("estimated_total_tokens", "sum"),
    estimated_cost_usd=("estimated_cost_usd", "sum"),
)
display(provider_estimates)
print("Estimated total cost across all enabled models and prompts: $", round(estimate_df["estimated_cost_usd"].sum(), 6))

,provider,model,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,openai,gpt-5.4-nano,direct_no_context,11,3154,880,4034,0.001731
1,openai,gpt-5.4-nano,definitions_only,11,22107,880,22987,0.005521
2,openai,gpt-5.4-nano,definitions_examples_no_metadata,11,33382,880,34262,0.007776
3,openai,gpt-5.4-nano,definitions_examples_with_metadata,11,39982,880,40862,0.009096
4,anthropic,claude-haiku-4-5-20251001,direct_no_context,11,4145,880,5025,0.008545
5,anthropic,claude-haiku-4-5-20251001,definitions_only,11,29486,880,30366,0.033886
6,anthropic,claude-haiku-4-5-20251001,definitions_examples_no_metadata,11,45116,880,45996,0.049516
7,anthropic,claude-haiku-4-5-20251001,definitions_examples_with_metadata,11,51079,880,51959,0.055479
8,gemini,gemini-2.5-flash-lite,direct_no_context,11,4145,880,5025,0.000767
9,gemini,gemini-2.5-flash-lite,definitions_only,11,29486,880,30366,0.003301


,provider,model,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,anthropic,claude-haiku-4-5-20251001,129826,3520,133346,0.147426
1,gemini,gemini-2.5-flash-lite,129826,3520,133346,0.014391
2,openai,gpt-5.4-nano,98625,3520,102145,0.024125


Estimated total cost across all enabled models and prompts: $ 0.185942


## 7. API wrappers

The wrappers return a standardized dictionary with:

- raw text response
- input/output/total tokens
- cost
- latency

The notebook uses the same prompt messages for all three providers.

In [ ]:
def extract_json_object(text: str) -> dict:
    """Parse model output into JSON, with fallbacks for markdown fences or extra text."""
    if text is None:
        return {}

    s = str(text).strip()
    if not s:
        return {}

    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE).strip()
    s = re.sub(r"\s*```$", "", s).strip()

    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            return obj
        if isinstance(obj, list) and obj and isinstance(obj[0], dict):
            return obj[0]
        return {}
    except Exception:
        pass

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            return obj if isinstance(obj, dict) else {}
        except Exception:
            return {}

    return {}


def normalize_model_prediction(parsed: dict, raw_text: str = "") -> str | float:
    candidate_keys = [
        "unlearning",
        "unlearning_label",
        "is_unlearning",
        "label",
        "prediction",
        "answer",
        "classification",
    ]

    if isinstance(parsed, dict):
        for key in candidate_keys:
            if key in parsed:
                norm = normalize_yes_no(parsed.get(key))
                if not pd.isna(norm):
                    return norm

    s = str(raw_text or "")

    field_match = re.search(
        r'"(?:unlearning|unlearning_label|is_unlearning|label|prediction|answer|classification)"\s*:\s*"?(yes|no|true|false)"?',
        s,
        flags=re.IGNORECASE,
    )
    if field_match:
        return normalize_yes_no(field_match.group(1))

    if len(s) < 200:
        yn_match = re.search(r"\b(yes|no)\b", s, flags=re.IGNORECASE)
        if yn_match:
            return normalize_yes_no(yn_match.group(1))

    return np.nan


class APIErrorForRetry(Exception):
    pass


def _retry_wait_from_error(error: Exception, default_wait: float) -> float:
    text = str(error)
    match = re.search(r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+(?:\.\d+)?)s", text)
    if match:
        return float(match.group(1)) + 1.0
    return default_wait


def retry_api_call(max_attempts=6, min_wait=5, max_wait=90):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(max_attempts):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    if attempt == max_attempts - 1:
                        raise
                    default_wait = min(max_wait, min_wait * (2 ** attempt))
                    wait_seconds = min(max_wait, _retry_wait_from_error(e, default_wait))
                    print(f"Retrying after error from {fn.__name__}: {e}\nWaiting {wait_seconds:.1f}s...")
                    time.sleep(wait_seconds)
            raise last_error
        return wrapper
    return decorator


@retry_api_call(max_attempts=3)
def call_openai(messages: list[dict], cfg: dict) -> dict:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv(cfg["api_key_env"]))
    start = time.time()
    try:
        response = client.chat.completions.create(
        model=cfg["model"],
        messages=messages,
        temperature=TEMPERATURE,
        max_completion_tokens=MAX_OUTPUT_TOKENS,
        response_format={"type": "json_object"},
    )
    except Exception as e:
        raise APIErrorForRetry(str(e))

    latency = time.time() - start
    raw_text = response.choices[0].message.content or ""
    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "prompt_tokens", None) or 0
    output_tokens = getattr(usage, "completion_tokens", None) or 0
    total_tokens = getattr(usage, "total_tokens", None) or input_tokens + output_tokens
    return {
        "raw_response": raw_text,
        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "total_tokens": int(total_tokens),
        "latency_seconds": latency,
    }


@retry_api_call(max_attempts=3, min_wait=5, max_wait=30)
def call_anthropic(messages: list[dict], cfg: dict) -> dict:
    from anthropic import Anthropic

    client = Anthropic(api_key=os.environ[cfg["api_key_env"]])

    system_text = ""
    anthropic_messages = []

    for msg in messages:
        role = msg.get("role")
        content = msg.get("content", "")

        if role == "system":
            system_text += content + "\n"
        elif role in {"user", "assistant"}:
            anthropic_messages.append({
                "role": role,
                "content": content,
            })

    response = client.messages.create(
        model=cfg["model"],
        max_tokens=MAX_OUTPUT_TOKENS,   # Anthropic uses max_tokens, not max_completion_tokens
        temperature=TEMPERATURE,
        system=system_text.strip(),
        messages=anthropic_messages,
    )

    raw_text = ""
    for block in response.content:
        if getattr(block, "type", None) == "text":
            raw_text += block.text

    input_tokens = getattr(response.usage, "input_tokens", 0)
    output_tokens = getattr(response.usage, "output_tokens", 0)

    parsed = extract_json_object(raw_text)
    pred = normalize_model_prediction(parsed, raw_text)

    return {
        "raw_text": raw_text,
        "parsed": parsed,
        "pred_unlearning": pred,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "status": "ok" if not pd.isna(pred) else "parse_failed",
    }


@retry_api_call(max_attempts=6, min_wait=8, max_wait=120)
def call_gemini(messages: list[dict], cfg: dict) -> dict:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=os.getenv(cfg["api_key_env"]))
    system_text = next((m["content"] for m in messages if m["role"] == "system"), "")
    user_text = "\n\n".join(m["content"] for m in messages if m["role"] == "user")

    start = time.time()
    try:
      response_schema = {
        "type": "object",
        "properties": {
            "unlearning": {"type": "string", "enum": ["Yes", "No"]},
            "confidence": {"type": "number"},
            "rationale": {"type": "string"},
        },
        "required": ["unlearning", "confidence", "rationale"],
    }

      response = client.models.generate_content(
        model=cfg["model"],
        contents=user_text,
        config=types.GenerateContentConfig(
            system_instruction=system_text,
            temperature=TEMPERATURE,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            response_mime_type="application/json",
            response_schema=response_schema,
        ),
    )
    except Exception as e:
        raise APIErrorForRetry(str(e))

    latency = time.time() - start
    raw_text = response.text or ""
    usage = getattr(response, "usage_metadata", None)
    input_tokens = getattr(usage, "prompt_token_count", None) or 0
    output_tokens = getattr(usage, "candidates_token_count", None) or 0
    total_tokens = getattr(usage, "total_token_count", None) or input_tokens + output_tokens
    return {
        "raw_response": raw_text,
        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "total_tokens": int(total_tokens),
        "latency_seconds": latency,
    }


def call_model(messages: list[dict], cfg: dict) -> dict:
    provider = cfg["provider"].lower()
    if provider == "openai":
        return call_openai(messages, cfg)
    if provider == "anthropic":
        return call_anthropic(messages, cfg)
    if provider == "gemini":
        return call_gemini(messages, cfg)
    raise ValueError(f"Unsupported provider: {provider}")

## 8. Run the A/B test grid

This cell writes a JSONL result after every API call, so you can safely interrupt and resume.

Set `RUN_API_CALLS = True` in the configuration cell when ready.

In [ ]:
RESULTS_JSONL = OUTPUT_DIR / "ab_test_raw_results.jsonl"

if RESULTS_JSONL.exists():
    clean_records = []

    with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)

            if rec.get("status") == "ok":
                clean_records.append(rec)

    with open(RESULTS_JSONL, "w", encoding="utf-8") as f:
        for rec in clean_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Kept {len(clean_records)} successful rows and removed failed/dry-run rows.")

RESUME_FROM_JSONL = True
RUN_API_CALLS = True

Kept 53 successful rows and removed failed/dry-run rows.


In [ ]:
RESULTS_JSONL = OUTPUT_DIR / "ab_test_raw_results.jsonl"


def make_run_key(provider: str, model: str, prompt_variant: str, row_id: str) -> str:
    s = f"{provider}|{model}|{prompt_variant}|{row_id}"
    return hashlib.md5(s.encode("utf-8")).hexdigest()


def load_existing_results(path: Path) -> tuple[list[dict], set[str]]:
    records = []
    keys = set()

    if not path.exists():
        return records, keys

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            rec = json.loads(line)

            # Do not resume from dry-run rows.
            if rec.get("status") == "dry_run":
                continue

            records.append(rec)

            if rec.get("status") == "ok" and "run_key" in rec:
              keys.add(rec["run_key"])

    return records, keys

def append_jsonl(path: Path, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def run_ab_test_grid(test_rows: pd.DataFrame, model_configs: list[dict], variants: list[str]) -> pd.DataFrame:
    existing_records, existing_keys = load_existing_results(RESULTS_JSONL) if RESUME_FROM_JSONL else ([], set())
    print(f"Loaded {len(existing_records)} existing records from {RESULTS_JSONL}.")

    records = existing_records.copy()
    total_calls = len(test_rows) * len(model_configs) * len(variants)

    if not RUN_API_CALLS:
        print("RUN_API_CALLS=False, so this cell will create a dry-run plan only and will not call APIs.")

    with tqdm(total=total_calls, desc="A/B grid") as pbar:
        for cfg in model_configs:
            for variant in variants:
                for _, row in test_rows.iterrows():
                    row_id = str(row.get("Number", row.name))
                    run_key = make_run_key(cfg["provider"], cfg["model"], variant, row_id)
                    if run_key in existing_keys:
                        pbar.update(1)
                        continue

                    messages = build_prompt_messages(variant, row)
                    estimated_input_tokens = estimate_messages_tokens(messages, provider=cfg["provider"])
                    base_record = {
                        "run_key": run_key,
                        "timestamp_utc": datetime.utcnow().isoformat(),
                        "provider": cfg["provider"],
                        "model": cfg["model"],
                        "prompt_variant": variant,
                        "row_id": row_id,
                        "document": row.get("Document", ""),
                        "text_content": row.get("Text Content", ""),
                        "ground_truth_unlearning": row.get("ground_truth_unlearning", np.nan),
                        "estimated_input_tokens": int(estimated_input_tokens),
                    }

                    if not RUN_API_CALLS:
                        rec = {
                            **base_record,
                            "status": "dry_run",
                            "raw_response": "",
                            "pred_unlearning": np.nan,
                            "confidence": np.nan,
                            "rationale": "",
                            "input_tokens": int(estimated_input_tokens),
                            "output_tokens": 0,
                            "total_tokens": int(estimated_input_tokens),
                            "cost_usd": cost_from_tokens(estimated_input_tokens, 0, cfg),
                            "latency_seconds": 0.0,
                            "parse_error": "",
                        }
                    else:
                        try:
                            api_result = call_model(messages, cfg)
                            parsed = extract_json_object(api_result["raw_response"])
                            pred = normalize_model_prediction(parsed, api_result["raw_response"])
                            confidence = parsed.get("confidence", np.nan) if isinstance(parsed, dict) else np.nan
                            rationale = parsed.get("rationale", "") if isinstance(parsed, dict) else ""
                            input_tokens = api_result["input_tokens"] or estimated_input_tokens
                            output_tokens = api_result["output_tokens"]
                            total_tokens = api_result["total_tokens"] or input_tokens + output_tokens
                            cost_usd = cost_from_tokens(input_tokens, output_tokens, cfg)
                            rec = {
                                **base_record,
                                "status": "ok" if not pd.isna(pred) else "parse_failed",
                                "raw_response": api_result["raw_response"],
                                "pred_unlearning": pred,
                                "confidence": confidence,
                                "rationale": rationale,
                                "input_tokens": int(input_tokens),
                                "output_tokens": int(output_tokens),
                                "total_tokens": int(total_tokens),
                                "cost_usd": cost_usd,
                                "latency_seconds": api_result["latency_seconds"],
                                "parse_error": "" if not pd.isna(pred) else "Could not parse Yes/No from model response.",
                            }
                        except Exception as e:
                            rec = {
                                **base_record,
                                "status": "error",
                                "raw_response": "",
                                "pred_unlearning": np.nan,
                                "confidence": np.nan,
                                "rationale": "",
                                "input_tokens": int(estimated_input_tokens),
                                "output_tokens": 0,
                                "total_tokens": int(estimated_input_tokens),
                                "cost_usd": cost_from_tokens(estimated_input_tokens, 0, cfg),
                                "latency_seconds": 0.0,
                                "parse_error": str(e),
                            }

                    append_jsonl(RESULTS_JSONL, rec)
                    records.append(rec)
                    existing_keys.add(run_key)
                    pbar.update(1)
                    time.sleep(cfg.get("request_sleep_seconds", REQUEST_SLEEP_SECONDS))

    return pd.DataFrame(records)


# Budget guard before real calls
active_cfgs = active_model_configs(require_api_key=RUN_API_CALLS)
preflight = estimate_full_grid(test_df, active_cfgs, PROMPT_VARIANTS_TO_RUN)
preflight_total_cost = preflight["estimated_cost_usd"].sum()
print("Preflight estimated cost: $", round(preflight_total_cost, 6))

if RUN_API_CALLS and preflight_total_cost > MAX_ESTIMATED_COST_USD:
    raise RuntimeError(
        f"Estimated cost ${preflight_total_cost:.4f} exceeds MAX_ESTIMATED_COST_USD=${MAX_ESTIMATED_COST_USD:.2f}. "
        "Increase the limit, reduce ROW_LIMIT, reduce MAX_FEWSHOT_EXAMPLES, or disable some models/prompts."
    )

results_df = run_ab_test_grid(test_df, active_cfgs, PROMPT_VARIANTS_TO_RUN)
print(results_df.shape)
display(results_df.head())

Preflight estimated cost: $ 0.185942
Loaded 53 existing records from outputs_unlearning_ab_test/ab_test_raw_results.jsonl.


A/B grid:   0%|          | 0/132 [00:00<?, ?it/s]

/tmp/ipykernel_22788/3843909512.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat(),


Retrying after error from call_gemini: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 36.870692652s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global'

KeyboardInterrupt: 

## 9. Evaluate predictions and summarize cost

This section creates:

- row-level prediction comparison
- prompt/model performance metrics
- token and cost summaries
- provider-level totals

In [ ]:
def label_to_binary(series: pd.Series) -> pd.Series:
    return series.apply(lambda x: 1 if normalize_yes_no(x) == "Yes" else (0 if normalize_yes_no(x) == "No" else np.nan))


def summarize_metrics(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[df["status"].eq("ok")].copy()
    scored["y_true"] = label_to_binary(scored["ground_truth_unlearning"])
    scored["y_pred"] = label_to_binary(scored["pred_unlearning"])
    scored = scored.dropna(subset=["y_true", "y_pred"])

    rows = []
    group_cols = ["provider", "model", "prompt_variant"]
    for keys, g in scored.groupby(group_cols):
        provider, model, variant = keys
        y_true = g["y_true"].astype(int)
        y_pred = g["y_pred"].astype(int)
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        accuracy = (tp + tn) / len(g) if len(g) else np.nan
        precision_yes = tp / (tp + fp) if (tp + fp) else np.nan
        recall_yes = tp / (tp + fn) if (tp + fn) else np.nan
        f1_yes = (2 * precision_yes * recall_yes / (precision_yes + recall_yes)) if (precision_yes + recall_yes) else np.nan
        rows.append({
            "provider": provider,
            "model": model,
            "prompt_variant": variant,
            "n_scored": len(g),
            "accuracy": accuracy,
            "precision_yes": precision_yes,
            "recall_yes": recall_yes,
            "f1_yes": f1_yes,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "input_tokens": g["input_tokens"].sum(),
            "output_tokens": g["output_tokens"].sum(),
            "total_tokens": g["total_tokens"].sum(),
            "cost_usd": g["cost_usd"].sum(),
            "avg_latency_seconds": g["latency_seconds"].mean(),
        })
    return pd.DataFrame(rows).sort_values(["f1_yes", "accuracy", "cost_usd"], ascending=[False, False, True]) if rows else pd.DataFrame()


correct_series = (
    results_df["ground_truth_unlearning"].apply(normalize_yes_no)
    == results_df["pred_unlearning"].apply(normalize_yes_no)
)
results_df["correct"] = correct_series.astype("object")
results_df.loc[results_df["status"] != "ok", "correct"] = np.nan

metrics_df = summarize_metrics(results_df)
usage_by_variant_df = results_df.groupby(["provider", "model", "prompt_variant", "status"], as_index=False).agg(
    n_rows=("row_id", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    total_tokens=("total_tokens", "sum"),
    cost_usd=("cost_usd", "sum"),
    avg_latency_seconds=("latency_seconds", "mean"),
)
usage_by_model_df = results_df.groupby(["provider", "model"], as_index=False).agg(
    n_records=("row_id", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    total_tokens=("total_tokens", "sum"),
    cost_usd=("cost_usd", "sum"),
)

print("Metrics by model/prompt:")
display(metrics_df)
print("Usage by variant:")
display(usage_by_variant_df)
print("Total usage by model:")
display(usage_by_model_df)

NameError: name 'results_df' is not defined

## 10. Save outputs

The Excel output contains separate sheets for row-level predictions, metrics, and usage summaries.

In [ ]:
PREDICTIONS_CSV = OUTPUT_DIR / "predictions_long.csv"
METRICS_CSV = OUTPUT_DIR / "metrics_by_model_prompt.csv"
USAGE_VARIANT_CSV = OUTPUT_DIR / "usage_by_variant.csv"
USAGE_MODEL_CSV = OUTPUT_DIR / "usage_by_model.csv"
OUTPUT_XLSX = OUTPUT_DIR / "unlearning_ab_test_results.xlsx"

results_df.to_csv(PREDICTIONS_CSV, index=False)
metrics_df.to_csv(METRICS_CSV, index=False)
usage_by_variant_df.to_csv(USAGE_VARIANT_CSV, index=False)
usage_by_model_df.to_csv(USAGE_MODEL_CSV, index=False)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    results_df.to_excel(writer, index=False, sheet_name="Predictions_Long")
    metrics_df.to_excel(writer, index=False, sheet_name="Metrics")
    usage_by_variant_df.to_excel(writer, index=False, sheet_name="Usage_By_Variant")
    usage_by_model_df.to_excel(writer, index=False, sheet_name="Usage_By_Model")
    estimate_df.to_excel(writer, index=False, sheet_name="Preflight_Estimates")

print("Saved:")
print("-", PREDICTIONS_CSV)
print("-", METRICS_CSV)
print("-", USAGE_VARIANT_CSV)
print("-", USAGE_MODEL_CSV)
print("-", OUTPUT_XLSX)

## 11. Inspect disagreements and parse failures

Use this section to understand where prompts/models fail and whether failures reflect true model errors or ambiguous labels.

In [ ]:
if "correct" in results_df.columns:
    disagreements = results_df[(results_df["status"] == "ok") & (results_df["correct"] == False)].copy()
    parse_failures = results_df[results_df["status"].isin(["parse_failed", "error"])].copy()

    print("Disagreements:", len(disagreements))
    display(disagreements[[
        "provider", "model", "prompt_variant", "row_id", "ground_truth_unlearning",
        "pred_unlearning", "confidence", "rationale", "text_content"
    ]].head(20))

    print("Parse/API failures:", len(parse_failures))
    display(parse_failures[[
        "provider", "model", "prompt_variant", "row_id", "status", "parse_error", "raw_response"
    ]].head(20))
else:
    print("No scored results yet. Set RUN_API_CALLS=True and run the grid.")

## 12. Recommended next steps after the first run

After you run the full test grid:

1. Select the best prompt based on F1 for `Unlearning = Yes`, not just accuracy.
2. Check false positives and false negatives manually.
3. If cost is too high, reduce few-shot examples or test a retrieval-based prompt that uses only the top-k most similar examples per paragraph.
4. Once the prompt is stable, process large batches in chunks and keep the JSONL resume behavior enabled.
5. Keep updating model prices in `MODEL_CONFIG`, because API pricing changes over time.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

RESULTS_JSONL = OUTPUT_DIR / "ab_test_raw_results.jsonl"

# Load existing successful results only
records = []
with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)

        # Keep only successful rows
        if rec.get("status") != "ok":
            continue

        # Skip Gemini
        if rec.get("provider") == "gemini":
            continue

        records.append(rec)

results_df = pd.DataFrame(records)

print("Loaded successful non-Gemini results:", len(results_df))
display(results_df.head())

Loaded successful non-Gemini results: 44


,run_key,timestamp_utc,provider,model,prompt_variant,row_id,document,text_content,ground_truth_unlearning,estimated_input_tokens,...,raw_response,pred_unlearning,confidence,rationale,input_tokens,output_tokens,total_tokens,cost_usd,latency_seconds,parse_error
0,1352714e9272898474f0d2cc888bee86,2026-07-03T12:52:16.533679,openai,gpt-5.4-nano,direct_no_context,14:2,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,170,...,"{""unlearning"":""No"",""confidence"":0.86,""rational...",No,0.86,The passage discusses organizational placement...,159,45,204,0.000088,2.717160,
1,f957dd9812cb60027aeab075ec3be1d4,2026-07-03T12:52:20.922444,openai,gpt-5.4-nano,direct_no_context,14:3,gao-06-442t.pdf,"In addition, we have done a great deal of work...",Yes,429,...,"{""unlearning"":""No"",""confidence"":0.78,""rational...",No,0.78,The passage describes reviewing and improving ...,414,45,459,0.000139,1.337355,
2,c8db4a35419ea894cf284216ec6be71e,2026-07-03T12:52:22.509241,openai,gpt-5.4-nano,direct_no_context,14:4,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,185,...,"{""unlearning"":""No"",""confidence"":0.86,""rational...",No,0.86,The passage discusses organizational placement...,174,45,219,0.000091,1.283314,
3,7d9d68608f7301cbfb222d4c8546b23c,2026-07-03T12:52:24.041832,openai,gpt-5.4-nano,direct_no_context,14:5,gao-06-442t.pdf,Because it is possible to respond to incidents...,Yes,230,...,"{""unlearning"":""No"",""confidence"":0.78,""rational...",No,0.78,The passage argues for a policy approach to ca...,217,54,271,0.000111,1.111421,
4,7c8f8479023b84f29b6dc84e6d2cb6f8,2026-07-03T12:52:25.405372,openai,gpt-5.4-nano,direct_no_context,14:6,gao-06-442t.pdf,A proactive approach to catastrophic disasters...,Yes,222,...,"{""unlearning"":""No"",""confidence"":0.86,""rational...",No,0.86,The passage discusses continuing and reinforci...,210,44,254,0.000097,1.041440,


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

RESULTS_JSONL = OUTPUT_DIR / "ab_test_raw_results.jsonl"

# Load existing successful results only
records2 = []
with open(RESULTS_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        rec = json.loads(line)

        records2.append(rec)

results_df2 = pd.DataFrame(records2)

display(results_df2.head())

,run_key,timestamp_utc,provider,model,prompt_variant,row_id,document,text_content,ground_truth_unlearning,estimated_input_tokens,...,raw_response,pred_unlearning,confidence,rationale,input_tokens,output_tokens,total_tokens,cost_usd,latency_seconds,parse_error
0,2d304b7e5effff9b67173ef9606fecb9,2026-07-03T12:27:48.780745,gemini,gemini-2.5-flash-lite,direct_no_context,14:2,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,210,...,"{""unlearning"": ""No"", ""confidence"": 0.9, ""ratio...",No,0.9,The passage discusses potential organizational...,157,47,206,0.000034,1.032465,
1,f1f63384ababdbeadfdacd8d7fd35458,2026-07-03T12:28:11.084090,gemini,gemini-2.5-flash-lite,direct_no_context,14:4,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,228,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage discusses potential organizational...,170,40,210,0.000033,0.849184,
2,9cb92b942702ec158d384288e12ea799,2026-07-03T12:28:37.344574,gemini,gemini-2.5-flash-lite,direct_no_context,14:6,gao-06-442t.pdf,A proactive approach to catastrophic disasters...,Yes,295,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage describes reinforcing existing rec...,209,31,240,0.000033,3.855061,
3,17430880b8451284a6ee8bf8141d041f,2026-07-03T12:28:50.623857,gemini,gemini-2.5-flash-lite,direct_no_context,14:8,gao-06-442t.pdf,Our work on interoperable communications ident...,Yes,426,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage focuses on identifying and solving...,298,39,339,0.000045,14.583380,
4,99ba6a7d903988364853ceda6929e5ab,2026-07-03T12:29:06.789669,gemini,gemini-2.5-flash-lite,direct_no_context,14:9,gao-06-442t.pdf,Damage and needs assessment: Damage and needs ...,Yes,272,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage describes a lack of preparedness a...,195,36,231,0.000034,0.931686,


In [ ]:
results_df2.shape

(97, 21)

In [ ]:
results_df2

,run_key,timestamp_utc,provider,model,prompt_variant,row_id,document,text_content,ground_truth_unlearning,estimated_input_tokens,...,raw_response,pred_unlearning,confidence,rationale,input_tokens,output_tokens,total_tokens,cost_usd,latency_seconds,parse_error
0,2d304b7e5effff9b67173ef9606fecb9,2026-07-03T12:27:48.780745,gemini,gemini-2.5-flash-lite,direct_no_context,14:2,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,210,...,"{""unlearning"": ""No"", ""confidence"": 0.9, ""ratio...",No,0.9,The passage discusses potential organizational...,157,47,206,0.000034,1.032465,
1,f1f63384ababdbeadfdacd8d7fd35458,2026-07-03T12:28:11.084090,gemini,gemini-2.5-flash-lite,direct_no_context,14:4,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,228,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage discusses potential organizational...,170,40,210,0.000033,0.849184,
2,9cb92b942702ec158d384288e12ea799,2026-07-03T12:28:37.344574,gemini,gemini-2.5-flash-lite,direct_no_context,14:6,gao-06-442t.pdf,A proactive approach to catastrophic disasters...,Yes,295,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage describes reinforcing existing rec...,209,31,240,0.000033,3.855061,
3,17430880b8451284a6ee8bf8141d041f,2026-07-03T12:28:50.623857,gemini,gemini-2.5-flash-lite,direct_no_context,14:8,gao-06-442t.pdf,Our work on interoperable communications ident...,Yes,426,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage focuses on identifying and solving...,298,39,339,0.000045,14.583380,
4,99ba6a7d903988364853ceda6929e5ab,2026-07-03T12:29:06.789669,gemini,gemini-2.5-flash-lite,direct_no_context,14:9,gao-06-442t.pdf,Damage and needs assessment: Damage and needs ...,Yes,272,...,"{""unlearning"":""No"",""confidence"":0.9,""rationale...",No,0.9,The passage describes a lack of preparedness a...,195,36,231,0.000034,0.931686,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,2d7644b0c56dd5f4f4fc7b99c2fef86a,2026-07-03T13:03:11.221089,anthropic,claude-haiku-4-5-20251001,definitions_examples_with_metadata,14:8,gao-06-442t.pdf,Our work on interoperable communications ident...,Yes,4693,...,,NaN,NaN,,4693,0,4693,0.004693,0.000000,'raw_response'
93,4196981e90e8cd44f707e76187d2d521,2026-07-03T13:03:13.154718,anthropic,claude-haiku-4-5-20251001,definitions_examples_with_metadata,14:9,gao-06-442t.pdf,Damage and needs assessment: Damage and needs ...,Yes,4539,...,,NaN,NaN,,4539,0,4539,0.004539,0.000000,'raw_response'
94,f785bc5062e03d1f274144351112fc15,2026-07-03T13:03:14.872916,anthropic,claude-haiku-4-5-20251001,definitions_examples_with_metadata,14:12,gao-06-442t.pdf,Because of FEMA’s mission performance during H...,Yes,5160,...,,NaN,NaN,,5160,0,5160,0.005160,0.000000,'raw_response'
95,c2e889f5e8720d9a92ec1064a34cb046,2026-07-03T13:03:16.874123,anthropic,claude-haiku-4-5-20251001,definitions_examples_with_metadata,14:13,gao-06-442t.pdf,The health care infrastructure in the New Orle...,Yes,4519,...,,NaN,NaN,,4519,0,4519,0.004519,0.000000,'raw_response'


In [ ]:
results_df2.to_csv("predictions_most.csv", index=False)

In [ ]:
# Normalize labels
def normalize_binary_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()

    yes_values = {"yes", "y", "true", "1", "unlearning"}
    no_values = {"no", "n", "false", "0", "not unlearning", "non-unlearning"}

    if s in yes_values:
        return "Yes"
    if s in no_values:
        return "No"

    if "not unlearning" in s or "non-unlearning" in s:
        return "No"
    if "unlearning" in s:
        return "Yes"

    return np.nan


# Make sure ground truth is available
# Your GPT Test sheet should already be loaded as test_df or gpt_test_df.
# If your variable is named differently, change test_df below.

ground_truth_df = test_df.copy()

ground_truth_df["ground_truth_unlearning_norm"] = ground_truth_df["Unlearning"].apply(normalize_binary_label)

# Use Number as the safest join key if available
if "Number" in results_df.columns and "Number" in ground_truth_df.columns:
    analysis_df = results_df.merge(
        ground_truth_df[["Number", "Text Content", "Unlearning", "ground_truth_unlearning_norm"]],
        on="Number",
        how="left"
    )
else:
    # fallback if Number was not saved in results
    analysis_df = results_df.copy()
    if "row_index" in analysis_df.columns:
        analysis_df = analysis_df.merge(
            ground_truth_df.reset_index().rename(columns={"index": "row_index"})[
                ["row_index", "Text Content", "Unlearning", "ground_truth_unlearning_norm"]
            ],
            on="row_index",
            how="left"
        )

analysis_df["pred_unlearning_norm"] = analysis_df["pred_unlearning"].apply(normalize_binary_label)
analysis_df["is_correct"] = analysis_df["pred_unlearning_norm"] == analysis_df["ground_truth_unlearning_norm"]

print("Rows available for analysis:", len(analysis_df))
display(analysis_df.head())

KeyError: 'ground_truth_unlearning_norm'